In [42]:
import os
import pickle
import streamlit as st
from dotenv import load_dotenv
from sqlalchemy import create_engine, inspect
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from PyPDF2 import PdfReader
import re

# ------------------- Config -------------------
"""load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
st.title("Argo SQL Chatbot (Region-Aware)")"""

# ------------------- File Paths -------------------
pdf_metadata_path = "floaat_merged.pdf"
faiss_file_path = "faiss_metadata.pkl"

import os
import pandas as pd
from sqlalchemy import create_engine, inspect

# ------------------- PostgreSQL Config -------------------
pg_user = os.getenv("POSTGRES_USER", "postgres")          # default postgres user
pg_pass = os.getenv("POSTGRES_PASSWORD", "harshavardhan7$")  # default password
pg_db   = os.getenv("POSTGRES_DB", "argo_db")              # default db name
pg_host = os.getenv("POSTGRES_HOST", "localhost")
pg_port = os.getenv("POSTGRES_PORT", "5432")

table_name = "new_argo"

# Connection URI
postgres_uri = f"postgresql+psycopg2://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}"

# Create SQLAlchemy engine
engine = create_engine(postgres_uri)

# Inspector (to inspect database structure)
inspector = inspect(engine)

print(f"✅ Connected to PostgreSQL database: {pg_db} at {pg_host}:{pg_port}")


# ------------------- Step 1: Load FAISS from Metadata PDF -------------------
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
if os.path.exists(faiss_file_path):
    with open(faiss_file_path, "rb") as f:
        vectorstore = pickle.load(f)
    st.info("ℹ️ Loaded FAISS index from metadata PDF.")
else:
    if os.path.exists(pdf_metadata_path):
        reader = PdfReader(pdf_metadata_path)
        raw_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                raw_text += text + "\n"
        blocks = re.split(r'\}\s*', raw_text)
        docs = []
        for block in blocks:
            block = block.strip()
            if block and not block.endswith("}"):
                block += "}"
            if block:
                docs.append(Document(page_content=block))
        vectorstore = FAISS.from_texts([d.page_content for d in docs], embedding_model)
        with open(faiss_file_path, "wb") as f:
            pickle.dump(vectorstore, f)
        st.success("✅ Created FAISS index from metadata PDF.")
    else:
        st.error("❌ Metadata PDF not found!")

query = st.text_input("Ask a question (natural language → SQL):")
if query and 'vectorstore' in locals():
    try:
        retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
        columns = [col["name"] for col in inspector.get_columns(table_name)]

        # ------------------- SQL Generation Prompt -------------------
        sql_template = f"""
        You are an expert SQL generator for the Argo dataset. Rules:
        1. Only use the following table and columns:
           Table: '{table_name}'
           Columns: {columns}
        2. Use context from metadata retrieved from FAISS. Do NOT invent any columns.
        3. SQL must:
           - Be a single statement ending with semicolon
           - Use SELECT, DISTINCT, WHERE, AND, OR, GROUP BY, ORDER BY, LIMIT
           - Use COUNT, SUM, AVG, MIN, MAX for aggregates
        4. Filters:
           - Numeric: =, BETWEEN, MIN, MAX, AVG, SUM
           - Text: =, LIKE
           - Date/Timestamp: CAST(column AS DATE) BETWEEN 'YYYY-MM-DD' AND 'YYYY-MM-DD'
        5. Important:
           - Include latitude/longitude filters only if the user mentions a region by name.
           - Combine multiple regions using OR if multiple mentioned.
           - Ensure AND/OR precedence is correct
           - Use PostgreSQL syntax for dates (e.g., TO_CHAR(CAST(juld AS DATE), 'YYYY-MM'))
           - For any column not in GROUP BY, wrap it in an aggregate function to avoid GroupingError
        Context: {{context}}
        User question: {{question}}
        SQL Query:
        """
        sql_prompt = PromptTemplate(input_variables=["context", "question"], template=sql_template)

        # ------------------- LLM Setup -------------------
        
        llm = ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            temperature=0,
            google_api_key=api_key
        )

        qa = RetrievalQA.from_chain_type(
            llm=llm,
            retriever=retriever,
            return_source_documents=True,
            chain_type_kwargs={"prompt": sql_prompt},
            input_key="question"
        )

        result = qa({"question": query})
        sql = result.get("result", "").strip()

        # ---- CLEAN SQL ----
        sql = re.sub(r"^```[a-zA-Z]*", "", sql)
        sql = sql.replace("```", "").strip()
        if sql.lower().startswith("sql"):
            sql = sql[3:].strip()

        if not sql:
            st.error("LLM did not generate SQL.")
        else:
            st.subheader("Generated SQL")
            st.code(sql, language="sql")

            # ---- Execute SQL in PostgreSQL ----
            try:
                res = pd.read_sql_query(sql, engine)
                if res.empty:
                    # If no rows, let LLM answer based on metadata
                    summary_template = f"""
                    You are an expert on the Argo dataset. The user asked:
                    '{{query}}'
                    The database returned no rows. Using your metadata knowledge from FAISS,
                    explain or answer the question as best as possible.
                    """
                    summary_prompt = PromptTemplate(input_variables=["query"], template=summary_template)
                    summary_text = llm.predict(summary_prompt.format(query=query))
                    st.info("ℹ️ No data in DB. LLM answer based on metadata:")
                    st.write(summary_text)
                else:
                    # Show result table
                    st.subheader("Query Result (first 10 rows)")
                    st.dataframe(res.head(10))

                    # Ask LLM to summarize the result
                    summary_template = f"""
                    You are an expert on the Argo dataset. The user asked:
                    '{{query}}'
                    Here is the query result in tabular form:
                    {{table}}
                    Summarize the result concisely for the user.
                    """
                    summary_prompt = PromptTemplate(input_variables=["query", "table"], template=summary_template)
                    summary_text = llm.predict(summary_prompt.format(query=query, table=res.head(10).to_dict()))
                    st.subheader("LLM Summary of Result")
                    st.write(summary_text)

            except Exception as e:
                st.error(f"❌ SQL execution failed: {e}")

    except Exception as e:
        st.error(f"❌ Error generating SQL: {e}")




from openai import OpenAI
from langchain.llms.base import LLM
from typing import Optional, List, Mapping, Any
query = st.text_input("Ask a question (natural language → SQL):")
if query and 'vectorstore' in locals():
    try:
        retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
        columns = [col["name"] for col in inspector.get_columns(table_name)]

        # ------------------- SQL Generation Prompt -------------------
        sql_template = f"""
        You are an expert SQL generator for the Argo dataset. Rules:
        1. Only use the following table and columns:
           Table: '{table_name}'
           Columns: {columns}
        2. Use context from metadata retrieved from FAISS. Do NOT invent any columns.
        3. SQL must:
           - Be a single statement ending with semicolon
           - Use SELECT, DISTINCT, WHERE, AND, OR, GROUP BY, ORDER BY, LIMIT
           - Use COUNT, SUM, AVG, MIN, MAX for aggregates
        4. Filters:
           - Numeric: =, BETWEEN, MIN, MAX, AVG, SUM
           - Text: =, LIKE
           - Date/Timestamp: CAST(column AS DATE) BETWEEN 'YYYY-MM-DD' AND 'YYYY-MM-DD'
        5. Important:
           - Include latitude/longitude filters only if the user mentions a region by name.
           - Combine multiple regions using OR if multiple mentioned.
           - Ensure AND/OR precedence is correct
           - Use PostgreSQL syntax for dates (e.g., TO_CHAR(CAST(juld AS DATE), 'YYYY-MM'))
           - For any column not in GROUP BY, wrap it in an aggregate function to avoid GroupingError
        Context: {{context}}
        User question: {{question}}
        SQL Query:
        """
        sql_prompt = PromptTemplate(input_variables=["context", "question"], template=sql_template)

        # ------------------- LLM Setup -------------------
        
        class OllamaLLM(LLM):
            def __init__(self, model: str, base_url: str, api_key: str, temperature: float = 0):
                self.model = model
                self.base_url = base_url
                self.api_key = api_key
                self.temperature = temperature
                self.client = OpenAI(base_url=base_url, api_key=api_key)
        
            def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=self.temperature,
                    stop=stop
                )
                return response.choices[0].message.content
        
            @property
            def _identifying_params(self) -> Mapping[str, Any]:
                return {"model": self.model}
        
            @property
            def _llm_type(self) -> str:
                return "ollama_llm"
        
        # ------------------- LLaMA LLM Setup -------------------
        MODEL = "llama3.2"
        llm = OllamaLLM(
            model=MODEL,
            base_url="http://localhost:11434/v1",
            api_key="ollama",
            temperature=0
        )

        qa = RetrievalQA.from_chain_type(
            llm=llm,
            retriever=retriever,
            return_source_documents=True,
            chain_type_kwargs={"prompt": sql_prompt},
            input_key="question"
        )

        result = qa({"question": query})
        sql = result.get("result", "").strip()

        # ---- CLEAN SQL ----
        sql = re.sub(r"^```[a-zA-Z]*", "", sql)
        sql = sql.replace("```", "").strip()
        if sql.lower().startswith("sql"):
            sql = sql[3:].strip()

        if not sql:
            st.error("LLM did not generate SQL.")
        else:
            st.subheader("Generated SQL")
            st.code(sql, language="sql")

            # ---- Execute SQL in PostgreSQL ----
            try:
                res = pd.read_sql_query(sql, engine)
                if res.empty:
                    # If no rows, let LLM answer based on metadata
                    summary_template = f"""
                    You are an expert on the Argo dataset. The user asked:
                    '{{query}}'
                    The database returned no rows. Using your metadata knowledge from FAISS,
                    explain or answer the question as best as possible.
                    """
                    summary_prompt = PromptTemplate(input_variables=["query"], template=summary_template)
                    summary_text = llm.predict(summary_prompt.format(query=query))
                    st.info("ℹ️ No data in DB. LLM answer based on metadata:")
                    st.write(summary_text)
                else:
                    # Show result table
                    st.subheader("Query Result (first 10 rows)")
                    st.dataframe(res.head(10))

                    # Ask LLM to summarize the result
                    summary_template = f"""
                    You are an expert on the Argo dataset. The user asked:
                    '{{query}}'
                    Here is the query result in tabular form:
                    {{table}}
                    Summarize the result concisely for the user.
                    """
                    summary_prompt = PromptTemplate(input_variables=["query", "table"], template=summary_template)
                    summary_text = llm.predict(summary_prompt.format(query=query, table=res.head(10).to_dict()))
                    st.subheader("LLM Summary of Result")
                    st.write(summary_text)

            except Exception as e:
                st.error(f"❌ SQL execution failed: {e}")

    except Exception as e:
        st.error(f"❌ Error generating SQL: {e}")

import pandas as pd
from sqlalchemy import create_engine

# PostgreSQL connection details
pg_user = "postgres"
pg_pass = "harshavardhan7$"
pg_db   = "argo_db"
pg_host = "localhost"       # or your DB host
pg_port = "5432"

# CSV file path
csv_file = "argo_data_export.csv"
table_name = "new_argo"

engine = create_engine(f"postgresql+psycopg2://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}")

df = pd.read_csv(csv_file)

# Drop 'id' column if it exists, because it's not in DB yet
df.drop(columns=["id"], errors="ignore", inplace=True)

# Automatically create table and insert data
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"✅ Imported {len(df)} rows into {table_name} table.")




from openai import OpenAI
from langchain.llms.base import LLM
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import pandas as pd
import re
from typing import Optional, List, Mapping, Any

# ------------------- LLaMA LLM Wrapper -------------------
from pydantic import BaseModel, Field
from openai import OpenAI
from langchain.llms.base import LLM
from typing import Optional, List, Mapping, Any

from pydantic import BaseModel, Field
from openai import OpenAI
from langchain.llms.base import LLM
from typing import Optional, List, Mapping, Any


class OllamaLLM(LLM, BaseModel):
    model: str = Field(...)
    base_url: str = Field(...)
    api_key: str = Field(...)
    temperature: float = Field(default=0.0)
    client: Optional[OpenAI] = Field(default=None, exclude=True)

    def __init__(self, **data: Any):
        super().__init__(**data)
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key)

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=self.temperature,
            stop=stop
        )
        return response.choices[0].message.content

    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {"model": self.model}

    @property
    def _llm_type(self) -> str:
        return "ollama_llm"

# ------------------- Set LLaMA Model -------------------
MODEL = "llama3.2"
llm = OllamaLLM(
    model=MODEL,
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    temperature=0
)

# ------------------- Build SQL Prompt -------------------
columns = [col["name"] for col in inspector.get_columns(table_name)]

sql_template = f"""
You are an expert SQL generator for the Argo dataset. Rules:
1. Only use the following table and columns:
   Table: '{table_name}'
   Columns: {columns}
2. Use context from metadata retrieved from FAISS. Do NOT invent any columns.
3. SQL must:
   - Be a single statement ending with semicolon
   - Use SELECT, DISTINCT, WHERE, AND, OR, GROUP BY, ORDER BY, LIMIT
   - Use COUNT, SUM, AVG, MIN, MAX for aggregates
4. Filters:
   - Numeric: =, BETWEEN, MIN, MAX, AVG, SUM
   - Text: =, LIKE
   - Date/Timestamp: CAST(column AS DATE) BETWEEN 'YYYY-MM-DD' AND 'YYYY-MM-DD'
5. Important:
   - Include latitude/longitude filters only if the user mentions a region by name.
   - Combine multiple regions using OR if multiple mentioned.
   - Ensure AND/OR precedence is correct
   - Use PostgreSQL syntax for dates (e.g., TO_CHAR(CAST(juld AS DATE), 'YYYY-MM'))
   - For any column not in GROUP BY, wrap it in an aggregate function to avoid GroupingError
Context: {{context}}
User question: {{question}}
SQL Query:
"""

sql_prompt = PromptTemplate(input_variables=["context", "question"], template=sql_template)

# ------------------- Query Setup -------------------
query = input("Enter your natural language question: ")

if query and 'vectorstore' in locals():
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    context = retriever.get_relevant_documents(query)
    context_text = "\n".join([doc.page_content for doc in context])

    # Generate SQL from LLaMA
    prompt_text = sql_prompt.format(context=context_text, question=query)
    sql = llm._call(prompt_text)

    # Clean SQL output
    sql = re.sub(r"^```[a-zA-Z]*", "", sql).replace("```", "").strip()
    if sql.lower().startswith("sql"):
        sql = sql[3:].strip()

    print("📝 Generated SQL:\n", sql)

    # Execute SQL in PostgreSQL
    try:
        res = pd.read_sql_query(sql, engine)
        if res.empty:
            print("ℹ️ No rows returned. Showing LLaMA answer based on metadata:")
            summary_template = f"""
            You are an expert on the Argo dataset. The user asked:
            '{{query}}'
            The database returned no rows. Using your metadata knowledge from FAISS,
            explain or answer the question as best as possible.
            """
            summary_prompt = PromptTemplate(input_variables=["query"], template=summary_template)
            summary_text = llm._call(summary_prompt.format(query=query))
            print(summary_text)
        else:
            print("\n✅ Query Result (first 10 rows):")
            display(res.head(10))

            summary_template = f"""
            You are an expert on the Argo dataset. The user asked:
            '{{query}}'
            Here is the query result in tabular form:
            {{table}}
            Summarize the result concisely for the user.
            """
            summary_prompt = PromptTemplate(input_variables=["query", "table"], template=summary_template)
            summary_text = llm._call(summary_prompt.format(query=query, table=res.head(10).to_dict()))
            print("\n💡 LLaMA Summary of Result:")
            print(summary_text)

    except Exception as e:
        print(f"❌ SQL execution failed: {e}")

from openai import OpenAI
from langchain.llms.base import LLM
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import pandas as pd
import re
from typing import Optional, List, Mapping, Any
from pydantic import BaseModel, Field

# ------------------- LLaMA LLM Wrapper -------------------
class OllamaLLM(LLM, BaseModel):
    model: str = Field(...)
    base_url: str = Field(...)
    api_key: str = Field(...)
    temperature: float = Field(default=0.0)
    client: Optional[OpenAI] = Field(default=None, exclude=True)

    def __init__(self, **data: Any):
        super().__init__(**data)
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key)

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=self.temperature,
            stop=stop
        )
        return response.choices[0].message.content

    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {"model": self.model}

    @property
    def _llm_type(self) -> str:
        return "ollama_llm"


# ------------------- Set LLaMA Model -------------------
MODEL = "llama3.2"
llm = OllamaLLM(
    model=MODEL,
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    temperature=0
)

# ------------------- Build SQL Prompt -------------------
columns = [col["name"] for col in inspector.get_columns(table_name)]
print(columns)
sql_template = f"""
You are an expert SQL generator for the Argo dataset stored in PostgreSQL.
Rules:
1. The table to query is named '{table_name}'.
2. The available columns are: {columns}.
3. Always reference columns exactly as they appear in the table (case-sensitive).
4. All filters must reference the correct table and column names.
5. Use context from metadata retrieved from FAISS; do NOT invent new columns.
6. SQL must:
   - Be a single statement ending with a semicolon
   - Use SELECT, DISTINCT, WHERE, AND, OR, GROUP BY, ORDER BY, LIMIT
   - Use COUNT, SUM, AVG, MIN, MAX for aggregates
7. Filters:
   - Numeric: =, BETWEEN, MIN, MAX, AVG, SUM
   - Text: =, LIKE
   - Date/Timestamp: CAST(column AS DATE) BETWEEN 'YYYY-MM-DD' AND 'YYYY-MM-DD'
8. Important:
   - Include latitude/longitude filters only if the user mentions a region by name.
   - Combine multiple regions using OR if multiple mentioned.
   - Ensure AND/OR precedence is correct.
   - Use PostgreSQL syntax for dates (e.g., TO_CHAR(CAST(juld AS DATE), 'YYYY-MM'))
   - For any column not in GROUP BY, wrap it in an aggregate function to avoid GroupingError.

Context: {{context}}
User question: {{question}}
SQL Query:
"""

sql_prompt = PromptTemplate(input_variables=["context", "question"], template=sql_template)


# ------------------- Query Setup -------------------
query = input("Enter your natural language question: ")

if query and 'vectorstore' in locals():
    try:
        # Retrieve context from FAISS
        retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
        context_docs = retriever.get_relevant_documents(query)
        context_text = "\n".join([doc.page_content for doc in context_docs])

        print("\n📡 FAISS Context Retrieved:\n", context_text)

        # Format prompt with context
        prompt_text = sql_prompt.format(context=context_text, question=query)

        # Generate SQL with LLaMA
        print("\n📡 Sending prompt to LLaMA...")
        sql = llm._call(prompt_text)

        # Clean SQL output
        sql = re.sub(r"^```[a-zA-Z]*", "", sql).replace("```", "").strip()
        if sql.lower().startswith("sql"):
            sql = sql[3:].strip()

        print("\n📝 Generated SQL:\n", sql)

        # Execute SQL in PostgreSQL
        try:
            res = pd.read_sql_query(sql, engine)
            if res.empty:
                print("\nℹ️ No rows returned. Showing LLaMA answer based on metadata:")
                summary_template = """
                You are an expert on the Argo dataset. The user asked:
                '{query}'
                The database returned no rows. Using your metadata knowledge from FAISS,
                explain or answer the question as best as possible.
                """
                summary_prompt = PromptTemplate(input_variables=["query"], template=summary_template)
                summary_text = llm._call(summary_prompt.format(query=query))
                print(summary_text)
            else:
                print("\n✅ Query Result (first 10 rows):")
                display(res.head(10))

                summary_template = """
                You are an expert on the Argo dataset. The user asked:
                '{query}'
                Here is the query result in tabular form:
                {table}
                Summarize the result concisely for the user.
                """
                summary_prompt = PromptTemplate(input_variables=["query", "table"], template=summary_template)
                summary_text = llm._call(summary_prompt.format(query=query, table=res.head(10).to_dict()))
                print("\n💡 LLaMA Summary of Result:")
                print(summary_text)

        except Exception as e:
            print(f"❌ SQL execution failed: {e}")

    except Exception as e:
        print(f"❌ Error generating SQL: {e}")

